In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import torch
import copy
from matplotlib.colors import hsv_to_rgb
import pandas as pd
import matplotlib as mpl
from matplotlib.widgets import LassoSelector
from matplotlib.path import Path
from sklearn.decomposition import PCA
from matplotlib.widgets import Slider

In [ ]:
data = pd.read_csv("\\\\NAS_LOCCO\\Amaury\\DATA\\4_polar_MFM_these\\2026_02_actin_moein_16.csv", delimiter=';')
frame = data['frame'].to_numpy().astype(int)
x = data['x'].to_numpy()
y = data['y'].to_numpy()
z = data['z'].to_numpy()
rho = data['rho'].to_numpy()
eta = data['eta'].to_numpy()
delta = data['delta'].to_numpy()
N_photons = data['N_photon'].to_numpy()
score = data['score'].to_numpy()
x_start = data['x_start'].to_numpy()
y_start = data['y_start'].to_numpy()
z_start = data['z_start'].to_numpy()
rho_start = data['rho_start'].to_numpy()
delta_start = data['delta_start'].to_numpy()

In [ ]:
%matplotlib qt
plt.scatter(x, y, c=frame, cmap='coolwarm', s=1)

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [8,3]
hist = plt.hist(score, bins=50)
plt.xlabel('Finale loss per PSF')
plt.ylabel('Occurences')

In [ ]:
plt.rcParams['figure.figsize'] = [8,3]
hist = plt.hist(N_photons, bins=80)
plt.xlim((0, 45000))
plt.xlabel('Photon number per PSF')
plt.ylabel('Occurences')

In [ ]:
plt.rcParams['figure.figsize'] = [8,3]
hist = plt.hist(z, bins=300)
plt.xlim((-500,3000))
plt.xlabel('z')
plt.ylabel('Occurences')

In [ ]:
loss_thresh = -250000
mask1 = (score<loss_thresh) & (delta<150) & (delta>0) & (N_photons>3000) & (N_photons<20000) & (z<800) & (z>0) & (eta>20) & (eta<160)
#mask1 = (score<loss_thresh) & (delta<150) & (N_photons>3000) & (N_photons<20000) & (z<1400) & (z>-250) & (eta>20) & (eta<160) 


threshold = (score<-430000) & (delta<150) & (x>17500) & (x<200000) & (y>2000) & (y<5500) & (z<1450) & (N_photons<85000) & (N_photons>6000) 
threshold2 = (score<-430000) & (delta<150) & (x>10000) & (x<22500) & (y>2000) & (y<15000) & (N_photons<85000) & (N_photons>6000) 
threshold3 = (score<-430000) & (delta<150) & (x>10000) & (x<22500) & (y>2000) & (y<15000) & (z<1450) & (N_photons<85000) & (N_photons>6000) 
threshold4 = (score<-430000) & (N_photons<85000) & (N_photons>6000) 

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [25,25]
plt.rcParams.update({'font.size': 15})
plt.style.use('default')
#plt.style.use('dark_background')
hues = rho[mask1] / 180.0
hsv_colors = np.stack((hues, np.ones_like(hues), np.ones_like(hues)), axis=1)
rgb_colors = hsv_to_rgb(hsv_colors)
plt.scatter(x[mask1]/1000, y[mask1]/1000, c=rgb_colors, s=0.05)
plt.axis('equal')
#plt.xlim((10000, 22500))
#plt.ylim((2200, 14000))
plt.xlabel('x ($\\mu$m)')
plt.ylabel('y ($\\mu$m)')

# Half-circle colorbar rotated 90° clockwise
center_x, center_y = 20, -10
radius = 2
angles = np.linspace(0, np.pi, 200)  # rotated arc
for i in range(len(angles)-1):
    theta1, theta2 = angles[i], angles[i+1]
    hue = (np.degrees(theta1)) / 180.0  # Map to 0-1 range
    color = hsv_to_rgb([hue, 1, 1])
    arc_x = [center_x + radius * np.cos(theta1), center_x + radius * np.cos(theta2)]
    arc_y = [center_y + radius * np.sin(theta1), center_y + radius * np.sin(theta2)]
    plt.plot(arc_x, arc_y, color=color, lw=8, solid_capstyle='butt')

# Tick labels with offset
tick_angles = [180, 90, 0]  # Corresponding to hue range
label_offset = [-6.,1, -5]
plt.text(center_x, center_y, "$\\rho$ ($\\degree$)", ha='center', va='center', fontsize=15)
for i, ang in enumerate(tick_angles):
    rad = np.radians(ang)
    tx = center_x + (radius + label_offset[i]) * np.cos(np.pi-rad)
    ty = center_y + (radius + label_offset[i]) * np.sin(np.pi-rad)
    plt.text(tx, ty, f"{(ang)}", ha='center', va='center', fontsize=15)
#plt.xlim((6, 13.5))
#plt.ylim((12.8, 18.5))
#plt.savefig("actin_rho2.png", format="png")
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [5,5]
hh = plt.hist(delta[mask1], bins=100)
print(len(delta[mask1]))

In [ ]:
plt.rcParams['figure.figsize'] = [3,3]
hh = plt.hist(eta, bins=100)
plt.axvline(90, c='r')
plt.grid()
plt.xlabel('$\\eta$ ($\\degree$)')
plt.ylabel('Occurrence')
#plt.savefig("eta1_global.svg", format="svg", dpi=300, bbox_inches='tight')

In [ ]:
plt.rcParams['figure.figsize'] = [3,3]
hh = plt.hist(N_photons[mask1], bins=100)
plt.grid()

In [ ]:
#%matplotlib qt
vals = delta[mask1]
plt.rcParams['figure.figsize'] = [7,7]
norm = mpl.colors.Normalize(vmin=0., vmax=180.)
orig_map=plt.cm.get_cmap('coolwarm')
reversed_map = orig_map.reversed()

sc = plt.scatter(
    x[mask1]/1000,
    y[mask1]/1000,
    c=vals,
    s=1,
    cmap='coolwarm',
    norm=norm
)

plt.axis('equal')
plt.xlabel('x ($\\mu$m)')
plt.ylabel('y ($\\mu$m)')

cbar = plt.colorbar(sc)
cbar.set_label('$\\delta$ ($\\degree$)')
#plt.savefig("actin_delta2.png", format="png", bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = [3, 3]
hh = plt.hist(rho[mask1], bins=100)

In [ ]:
x_ = rho[mask1]
y_ = delta[mask1]

# Sort by x (important for moving average)
order = np.argsort(x_)
x_ = x_[order]
y_ = y_[order]

# Sliding window size (10 degrees)
window = 10

x_avg = []
y_avg = []

for i in range(len(x_)):
    mask = (x_ >= x_[i] - window/2) & (x_ <= x_[i] + window/2)
    if np.sum(mask) > 0:
        x_avg.append(x_[i])
        y_avg.append(np.mean(y_[mask]))

# Plot
plt.scatter(x_, y_, s=0.01)
plt.plot(x_avg, y_avg, linewidth=2)
plt.grid()
plt.xlabel("rho (deg)")
plt.ylabel("delta")

# select  ROI

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = eta[mask1]
sc = ax.scatter(x[mask1] , y[mask1], c=vals , cmap='coolwarm', norm=norm, s=1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((x, y))
mask = np.zeros(len(x), dtype=bool) 

def onselect(verts):
    global mask
    path = Path(verts)
    mask = path.contains_points(points) & mask1
    print(mask)

lasso = LassoSelector(ax, onselect)
plt.show()

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=0., vmax=180.)
vals = eta[mask]
sc = ax.scatter(x[mask] , z[mask], c=vals , cmap='coolwarm', norm=norm, s=1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((x, z))
mask2 = np.zeros(len(x), dtype=bool) 

def onselect(verts):
    global mask2
    path = Path(verts)
    mask2 = path.contains_points(points) & mask
    print(mask2)

lasso = LassoSelector(ax, onselect)
plt.show()

## 3D plot of the ROI

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
norm = mpl.colors.Normalize(vmin=0., vmax=180.)
vals = eta[mask2]
sc = ax.scatter(x[mask2] , y[mask2], z[mask2], c=vals , cmap='coolwarm', norm=norm, s=1)
ax.axis('equal')
cb = plt.colorbar(sc)

## eta anlysis

In [ ]:
angles = np.deg2rad(eta[mask2])
bins = 18
counts, bin_edges = np.histogram(angles, bins=bins, density=True)

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
width = bin_edges[1] - bin_edges[0]

# Duplicate with π shift
bin_centers_full = np.concatenate([bin_centers, bin_centers + np.pi])
counts_full = np.concatenate([counts, counts])

# ----- Figure layout -----
fig = plt.figure(figsize=(4,4))

# Polar subplot
ax1 = fig.add_subplot(2, 1, 1, projection='polar')

ax1.bar(bin_centers_full, counts_full, width=width, alpha=0.8)

ax1.set_theta_zero_location("N")
ax1.set_theta_direction(-1)
ax1.set_yticklabels([])

# PCA
XY = np.column_stack((x[mask2], y[mask2]))

pca = PCA(n_components=1)
principal_coord = pca.fit_transform(XY).flatten()

# Sort along principal axis
idx = np.argsort(principal_coord)
x_sorted = principal_coord[idx]
z_sorted = z[mask2][idx]   # use .values if it's pandas

# Moving average window size (adjust!)
window = 20

z_smooth = np.convolve(
    z_sorted,
    np.ones(window)/window,
    mode='valid'
)

# Correct matching x values
x_smooth = x_sorted[:len(z_smooth)]
vals = eta[mask2]
# Cartesian subplot
ax2 = fig.add_subplot(2, 1, 2)
#ax2.plot(x_smooth, z_smooth, color='red', linewidth=2)
ax2.scatter(principal_coord, z[mask2], s=1, alpha=1. ,c=vals , cmap='coolwarm', norm=norm)

ax2.set_xlabel("lateral coordinate (nm)")
ax2.set_ylabel("z (nm)")
ax2.set_aspect('equal')
#ax2.set_title("lat–z scatter")
ax2.grid()
ax2.set_ylim(-300,1500)
plt.tight_layout()
#plt.savefig("actin2_zone1.png", format="png")
plt.show()

# Sliding window

In [ ]:
angle_to_analyse=eta

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15, 15]

# Example: angle array in degrees (replace with yours)
theta = angle_to_analyse[mask1]   # or use your own angle array

window_width = 10.0  # degrees
half_width = window_width / 2

fig, ax = plt.subplots()
plt.subplots_adjust(bottom=0.15)

norm = mpl.colors.Normalize(vmin=0., vmax=180.)
vals = angle_to_analyse

# Initial center
theta0 = 0.0

# Initial mask
def compute_mask(center):
    diff = (theta - center + 180) % 360 - 180
    return (np.abs(diff) <= half_width)&mask1

mask2 = compute_mask(theta0)

# Initial scatter
sc = ax.scatter(
    x[mask2],
    y[mask2],
    c=vals[mask2],
    cmap='coolwarm',
    norm=norm,
    s=1
)

ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()

# Slider axis
ax_slider = plt.axes([0.2, 0.05, 0.6, 0.03])
slider = Slider(
    ax=ax_slider,
    label='Angle center (deg)',
    valmin=half_width,
    valmax=180-half_width,
    valinit=theta0
)

# Update function
def update(val):
    center = slider.val
    mask2 = compute_mask(center)

    # Update scatter points
    sc.set_offsets(np.column_stack((x[mask2], y[mask2])))
    sc.set_array(vals[mask2])

    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()

In [ ]:
import tifffile

In [ ]:
with tifffile.TiffFile('\\\\NAS_LOCCO\\Amaury\\DATA\\4polar_data_raw\\2026_02_10_actin_Moein\\widefield\\Calib_Polar_2026-02-10\\images\\RAW_DATA\\image_Pos0.ome.tif') as tif:
    print(tif.pages[0].shape)
    raw = np.zeros((512,512))
    for i in range(10):
        raw += tif.pages[i].asarray()

In [ ]:
plt.imshow(raw, cmap='gray', vmin=2700, vmax=4200)
#plt.savefig("actin1_widefield.png", format="png")
plt.show()

# precision analysis

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]

fig = plt.figure()
ax = fig.add_subplot()

sc = ax.scatter(x[mask1], y[mask1], c=frame[mask1], cmap='coolwarm', s=1)

ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()

points = np.column_stack((x, y))

# list to store all masks
masks = []

def onselect(verts):
    path = Path(verts)
    
    current_mask = path.contains_points(points) & mask1
    
    masks.append(current_mask)
    
    print(f"Selection {len(masks)}: {current_mask.sum()} points")

lasso = LassoSelector(ax, onselect)

plt.show()

In [ ]:
stdx = []
stdy = []
stdz = []
stdrho = []
stdeta = []
stddelta = []
for u in range(len(masks)):
    stdx.append(np.std(x[masks[u]]))
    stdy.append(np.std(y[masks[u]]))
    stdz.append(np.std(z[masks[u]]))
    stdrho.append(np.std(rho[masks[u]]))
    stdeta.append(np.std(eta[masks[u]]))
    stddelta.append(np.std(delta[masks[u]]))
stdx = np.array(stdx)
stdy = np.array(stdy)
stdz = np.array(stdz)
stdrho = np.array(stdrho)
stdeta = np.array(stdeta)
stddelta = np.array(stddelta)

In [ ]:
print(np.mean(stdx))
print(np.mean(stdy))
print(np.mean(stdz))
print(np.mean(stdrho))
print(np.mean(stdeta))
print(np.mean(stddelta))

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [4,4]
hh = plt.hist(stdrho, bins=10)

In [ ]:
def recursive_call(i, x, y, z, rho, eta, delta, frame, previous_index):
    #print(i, frame[i]+1)
    mask = (frame==frame[i]+1) & ((x-x[i])**2+(y-y[i])**2<50**2) & ((z-z[i])<75) #& (np.abs(rho-rho[i])<30) & (np.abs(eta-eta[i])<30)
    if (mask==True).any():
        #print('recursion')
        already_counted[i] = False
        return recursive_call(np.where(mask)[0][0], x, y, z, rho, eta, delta, frame, previous_index=np.concatenate((previous_index, np.where(mask)[0])))
    else:
        return previous_index.astype(int)

In [ ]:
not_counted = np.zeros(len(x))
not_counted[:] = True
stdx = []
stdy = []
stdz = []
stdrho = []
stdeta = []
stddelta = []
for i in range(len(x)):
    if not_counted[i]:
        indices =  recursive_call(i, x, y, z, rho, eta, delta, frame, previous_index=np.array([i]))
        #print(i, indices)
        if len(indices)>1:
            #print(indices)
            stdx.append(np.std(x[indices]))
            stdy.append(np.std(y[indices]))
            stdz.append(np.std(z[indices]))
            stdrho.append(np.std(rho[indices]))
            stdeta.append(np.std(eta[indices]))
            stddelta.append(np.std(delta[indices]))

            x[indices[-1]] = np.mean(x[indices])
            x[indices[:-1]] = np.nan
            y[indices[-1]] = np.mean(y[indices])
            y[indices[:-1]] = np.nan
            z[indices[-1]] = np.mean(z[indices])
            z[indices[:-1]] = np.nan
            rho[indices[-1]] = np.mean(rho[indices])
            rho[indices[:-1]] = np.nan
            eta[indices[-1]] = np.mean(eta[indices])
            eta[indices[:-1]] = np.nan
            delta[indices[-1]] = np.mean(delta[indices])
            delta[indices[:-1]] = np.nan
            N_photons[indices[-1]] = np.mean(N_photons[indices])
            N_photons[indices[:-1]] = np.nan
            score[indices[-1]] = np.mean(score[indices])
            score[indices[:-1]] = np.nan
            frame[indices[-1]] = np.mean(frame[indices])
            frame[indices[:-1]] = np.nan
print('removed ', len(np.where(np.isnan(x))[0]), ' over ', len(x))

In [ ]:
x = x[~np.isnan(x)]
y = y[~np.isnan(y)]
z = z[~np.isnan(z)]
rho = rho[~np.isnan(rho)]
eta = eta[~np.isnan(eta)]
delta = delta[~np.isnan(delta)]
N_photons = N_photons[~np.isnan(N_photons)]
score = score[~np.isnan(score)]
frame = frame[~np.isnan(frame)]

In [ ]:
hh = plt.hist(stdx, bins=50)

In [ ]:
print(np.mean(stdx))
print(np.mean(stdy))
print(np.mean(stdz))
print(np.mean(stdrho))
print(np.mean(stdeta))
print(np.mean(stddelta))

In [ ]:
mask1 = (score<loss_thresh) & (delta<150) & (delta>0) & (N_photons>3000) & (N_photons<20000) & (z<900) & (z>-200) & (eta>20) & (eta<160)

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]

fig = plt.figure()
ax = fig.add_subplot()

sc = ax.scatter(x[mask1], y[mask1], c=eta[mask1], cmap='coolwarm', s=1)

ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()

points = np.column_stack((x, y))

# list to store all masks
masks = []

def onselect(verts):
    path = Path(verts)
    
    current_mask = path.contains_points(points) & mask1
    
    masks.append(current_mask)
    
    print(f"Selection {len(masks)}: {current_mask.sum()} points")

lasso = LassoSelector(ax, onselect)

plt.show()